# RL-QAS: <ins>R</ins>einforcement <ins>L</ins>earning for <ins>Q</ins>uantum <ins>A</ins>rchitecture <ins>S</ins>earch

<p align="center">
  <img src="pics/intro.png" alt="title">
</p>

## Importing the dependencies

### Import quantum simulator

In [3]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity

### Importing modules to train neural network

In [5]:
import torch
import torch.nn as nn
import random
import copy
from collections import namedtuple
import numpy as np
from itertools import product

----------------------
# RL: <ins>R</ins>einforcement <ins>L</ins>earning
## The framework

<p align="center">
  <img src="pics/RL.png" alt="title">
</p>

----------------------
# RL-QAS: <ins>R</ins>einforcement <ins>L</ins>earning for <ins>Q</ins>uantum <ins>A</ins>rchitecture <ins>S</ins>earch
## The framework

<p align="center">
  <img src="pics/RL_QAS.png" alt="title">
</p>

## Action space preparation

**Action space** $\rightarrow$ **Quantum gates**. We consider the universal gateset:

- 1-qubit gates: $\texttt{RX}$, $\texttt{RY}$, $\texttt{RZ}$,
- 2-qubit gates: $\texttt{CNOT}$

The **action space** is represented by a *dictionary* where each *key* in the dictionary is a quantum gate represented as:

$\begin{equation} \textrm{action space dict} = \{0:[\underbrace{0,1}_{\text{position of 2-qubit gate}},\underbrace{N,0}_{\text{position of 1-qubit gate}}], 1: [N,0, 0,0], ...\} \end{equation}$

### Disecting an action
The 0th key action defines: $[\underbrace{0}_{\text{position of control}}, \underbrace{1}_{\text{position of target}}, \underbrace{N}_{\text{1-qubit rotation position}}, \underbrace{0}_{\text{1-qubit rotation direction}}]$

- Hence the 0-th key: $[0, 1, N, 0]$ defines a $\texttt{CNOT}$ gate with control at 0th qubit and target at 1st qubit. The 3rd and 4th terms are place holders and having an $N$ at 3rd position means it is not a 1-qubit gate. So each action is basicallt one gate
- The 1st key: $[N,0, 0,0]$, having an $N$ in the control position defines that it is not representing a 2-qubit gate. And then we have 3rd element says we have 1-qubit gate on 1st qubit and the 4th element says we have a $\texttt{X}$ gate.

<p align="center">
  <img src="pics/action_space_mod.png" alt="title">
</p>

In [ ]:
def dictionary_of_actions(num_qubits):
    """
    Creates dictionary of actions for system which steers positions of gates,
    and axes of rotations.
    """
    dictionary = dict()
    i = 0
         
    for c, x in product(range(num_qubits), range(num_qubits)):
        if c != x:
            dictionary[i] = [c, x, num_qubits, 0]
            i += 1
   
    """h  denotes 1q gate. 0, 1, 2, 3 -->  X, Y, Z, H gate """
    for r, h in product(range(num_qubits),
                           range(0, 4)):
        dictionary[i] = [num_qubits, 0, r, h]
        i += 1
    return dictionary

## Here is how the dictionary of the actions looks like

In [7]:
dictionary_of_actions(2)

{0: [0, 1, 2, 0],
 1: [1, 0, 2, 0],
 2: [2, 0, 0, 0],
 3: [2, 0, 0, 1],
 4: [2, 0, 0, 2],
 5: [2, 0, 0, 3],
 6: [2, 0, 1, 0],
 7: [2, 0, 1, 1],
 8: [2, 0, 1, 2],
 9: [2, 0, 1, 3]}

### Action space design:

> Can you edit the dictionary of actions to remove the Y and Z gates and construct a new function called "dictionary_of_actions_h_x_cx"? 

> After doing the first step: Can you remove H gate and add S gate to the dictionary of actions in a new function called "dictionary_of_actions_remove_s_x_cx"?

Rememebr if you modify this you must also modify the environment:

- The state size and 
- The "_make_circuit" function.

> After successfully completing first two steps:

- Train and compare agent with action space *(a1)* "X, Y, Z, H, CX" *(a2)* "X, H, CX"
- Train and compare agent wit action space *(a1)* "X, Y, Z, H, CX" *(a3)* "X, S, CX"

> Investigate weather with *(a1)* an agent consumes more/less training time and *(a2)*
> Investigate weather with *(a3)* the agent successfully can construct "SXS" which is equivalent to "H" gate to contruct maximally entangled state  

In [ ]:
def dictionary_of_actions_h_x_cx(num_qubits):
    """
    Creates dictionary of actions for system which steers positions of gates,
    and axes of rotations.
    """
    dictionary = dict()
    i = 0
         
    for c, x in product(range(num_qubits), range(num_qubits)):
        if c != x:
            dictionary[i] = [c, x, num_qubits, 0]
            i += 1
   
    """h  denotes 1q gate. 0, 1 -->  X, H gate """
    ## YOUR ANSWER HERE
    return dictionary

In [ ]:
def dictionary_of_actions_s_x_cx(num_qubits):
    """
    Creates dictionary of actions for system which steers positions of gates,
    and axes of rotations.
    """
    dictionary = dict()
    i = 0
         
    for c, x in product(range(num_qubits), range(num_qubits)):
        if c != x:
            dictionary[i] = [c, x, num_qubits, 0]
            i += 1
   
    """h  denotes 1q gate. 0, 1 -->  X, S gate """
    ## YOUR ANSWER HERE
    return dictionary